In [1]:
# install huggingface hub
!pip install huggingface_hub -q

# login to huggingface (create free account at huggingface.co first)
from huggingface_hub import login
login()  # it will ask for your token — get it from huggingface.co/settings/tokens

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [12]:
# upload your trained model to HuggingFace (free hosting)
from huggingface_hub import HfApi

api = HfApi()

# create a repo and upload your model
api.create_repo(repo_id="MUHAMMADSAADAMIN/polyguard-model", private=False, exist_ok=True)

api.upload_folder(
    folder_path="/content/drive/MyDrive/PolyGuard/03-models/polyguard_model",
    repo_id="MUHAMMADSAADAMIN/polyguard-model",
    repo_type="model"
)

print("Model uploaded! Your model is now at:")
print("https://huggingface.co/MUHAMMADSAADAMIN/polyguard-model")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...d_model/model.safetensors:   6%|6         | 32.0MB /  499MB            

Model uploaded! Your model is now at:
https://huggingface.co/MUHAMMADSAADAMIN/polyguard-model


### Verify Model Directory

The error message indicates that the path provided to `upload_folder` is not a valid directory. Let's check the contents of the specified path in your Google Drive to ensure it exists and is a folder.

In [13]:
import os

model_folder_path = "/content/drive/MyDrive/PolyGuard/03-models/polyguard_model"

# Mount Google Drive if not already mounted
from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

if os.path.exists(model_folder_path):
    if os.path.isdir(model_folder_path):
        print(f"Listing contents of '{model_folder_path}':")
        for item in os.listdir(model_folder_path):
            print(f"  - {item}")
    else:
        print(f"Error: '{model_folder_path}' exists but is not a directory.")
else:
    print(f"Error: '{model_folder_path}' does not exist. Please check the path in your Google Drive.")

Listing contents of '/content/drive/MyDrive/PolyGuard/03-models/polyguard_model':
  - model.safetensors
  - config.json
  - tokenizer_config.json
  - tokenizer.json


### Deploying to Hugging Face Spaces

It seems there might be some confusion between a Hugging Face *model repository* and a Hugging Face *Space*.

*   A **model repository** (`repo_type="model"`) is primarily used to store model weights, tokenizer files, and configuration, making your model shareable and loadable.
*   A **Space** (`repo_type="space"`) is a more comprehensive environment where you can host interactive machine learning demos (e.g., using Gradio or Streamlit), web applications, or even host your model for inference within a web interface. A Space usually contains an `app.py` or `app.ipynb` file, a `requirements.txt`, and your model artifacts (often within a subfolder like `models/` or `artifacts/`).

Your current code is attempting to upload a folder to a **model repository**.

Could you clarify if you want to:
1.  **Upload your model files to a model repository** (for sharing the model itself, as your current code attempts but with a directory path issue).
2.  **Create a Hugging Face Space application** that uses your model (this would involve creating an `app.py` file, `requirements.txt`, and then uploading all these files to a new Space repository with `repo_type="space"`).

### Preparing your Hugging Face Space

In [7]:
import os
import shutil

# Define the path for your Hugging Face Space local directory
space_name = "polyguard-space" # You can customize this
space_path = f"./{space_name}"

# Define the path where your model files are currently located
model_source_path = "/content/drive/MyDrive/PolyGuard/03-models/polyguard_model"

# Create the main space directory if it doesn't exist
os.makedirs(space_path, exist_ok=True)

# Create a subfolder for your model within the space directory
model_dest_path = os.path.join(space_path, "model")
os.makedirs(model_dest_path, exist_ok=True)

# Copy model files from Google Drive to the space's model subfolder
# Make sure the source directory exists and is valid
if os.path.isdir(model_source_path):
    print(f"Copying model files from '{model_source_path}' to '{model_dest_path}'...")
    for item_name in os.listdir(model_source_path):
        source_item = os.path.join(model_source_path, item_name)
        dest_item = os.path.join(model_dest_path, item_name)
        if os.path.isfile(source_item):
            shutil.copy2(source_item, dest_item)
    print("Model files copied successfully.")
else:
    print(f"Error: Source model path '{model_source_path}' is not a directory or does not exist.")

print(f"Hugging Face Space directory '{space_path}' created and model files copied.")

Copying model files from '/content/drive/MyDrive/PolyGuard/03-models/polyguard_model' to './polyguard-space/model'...
Model files copied successfully.
Hugging Face Space directory './polyguard-space' created and model files copied.


### Create `app.py` for your Space

This `app.py` will define the user interface and how your model is used. For this example, we'll use `Gradio` to create a simple web interface. You might need to adjust this code based on your specific model architecture and how it's loaded and run.

In [8]:
%%writefile {space_path}/app.py

import gradio as gr
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import os

# Define the model path within the Space
MODEL_PATH = "./model"

# Load your model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

# Create a Hugging Face pipeline
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

def predict_sentiment(text):
    result = classifier(text)[0]
    label = result['label']
    score = result['score']
    return f"Label: {label}, Score: {score:.4f}"

# Create the Gradio interface
iface = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(lines=5, placeholder="Enter text here..."),
    outputs="text",
    title="PolyGuard Model Demo",
    description="A simple Gradio interface to demonstrate the PolyGuard model."
)

iface.launch(share=True)

Writing ./polyguard-space/app.py


### Create `requirements.txt` for your Space

This file lists all the Python packages your `app.py` needs.

In [9]:
%%writefile {space_path}/requirements.txt

gradio
transformers
pytorch

Writing ./polyguard-space/requirements.txt


### Uploading your Space to Hugging Face

Now that you have your `app.py`, `requirements.txt`, and model files organized in the `{space_name}` directory, you can upload it to Hugging Face as a 'Space'.

In [11]:
from huggingface_hub import HfApi

api = HfApi()

repo_id = f"MUHAMMADSAADAMIN/{space_name}" # Or choose a new repo_id for your space

# Create the Hugging Face Space repository (if it doesn't exist)
# Note: `repo_type="space"` is crucial here
api.create_repo(repo_id=repo_id, private=False, repo_type="space", exist_ok=True, space_sdk='gradio')

# Upload the entire folder containing your app, requirements, and model
api.upload_folder(
    folder_path=space_path,
    repo_id=repo_id,
    repo_type="space",
    commit_message="Initial commit of PolyGuard Space"
)

print(f"Space uploaded! Your Hugging Face Space is now at:")
print(f"https://huggingface.co/spaces/{repo_id}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...e/model/model.safetensors:   3%|2         | 13.2MB /  499MB            

Space uploaded! Your Hugging Face Space is now at:
https://huggingface.co/spaces/MUHAMMADSAADAMIN/polyguard-space
